> **GitHub管理版（2026-11-05移管）**
> 既存の実験コードを移管したものです。セル出力は除去しており、今回の移管では実験を再実行していません。入力データ・保存済み結果は従来のDrive等を参照します。実行条件、認証、保存先を確認してから実行してください。元ファイルと用途は [ノートブック一覧](https://github.com/ryy1210/RMT_utils/blob/main/notebooks/README.md) を参照してください。

## 本番実行

上から順に実行してください。最初のセルでDriveを接続し、次のセルで環境変数または非表示入力で受け取ったAPIキーを`os.environ`へ設定します。
計算セルは行列ごとにCSVとスペクトルをDriveへ保存するため、切断後に同じ設定で再実行すると完了済み行列を飛ばして再開します。
全行列の計算、図表保存、W&B送信が完了すると、最終セルがDriveを同期してColabランタイムを自動的に解放します。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys
repo=Path('/content/RMT_utils')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/ryy1210/RMT_utils.git',str(repo)],check=True)
subprocess.run(['git','checkout','6739c2a0355a7e97336e0289176bd6f40aeb8877'],cwd=repo,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(repo/'notebooks/requirements-bulk.txt')],check=True)
print('ENVIRONMENT_READY')


In [ ]:
# 認証情報はセルに保存しない。環境変数がなければ非表示の入力欄で指定する。
# このNotebook専用ランタイムへAPIキーを明示的に設定します。
import os

os.environ["HF_TOKEN"] = (__import__("os").environ.get("HF_TOKEN") or __import__("getpass").getpass("HF_TOKEN: ").strip())
os.environ["WANDB_API_KEY"] = (__import__("os").environ.get("WANDB_API_KEY") or __import__("getpass").getpass("WANDB_API_KEY: ").strip())

assert os.environ["HF_TOKEN"].startswith("hf_")
assert os.environ["WANDB_API_KEY"].startswith("wandb_v1_")
print("HF_TOKEN and WANDB_API_KEY were assigned to os.environ.")


# MPバルクの整合性診断 — Llama 3B / Gemma 4B
## 実行状態
本番実行用に設定済みです。実行結果はまだ保存されていません。
## 方法
保存済み分散・閾値を固定し、λ=s²/n ≤ 閾値の固有値群を総質量1に再正規化してMPと比較します。
KSはp値ではありません。小さいbulkKSだけで真のシグナル分離やPPL維持は証明できません。
DE分散1の条件は独自のMP上端+TW閾値を用いる補助解析です。

## 環境
VS CodeのGoogle Colab拡張でNotebookを開き、Colabカーネルへ接続できます。
最初はCPUで人工行列テスト。モデル実験はまずT4で少数行列を計測し、必要なときだけL4等を検討してください。
A100への自動切替はありません。ランタイムは手動で選択します。
Colabではリポジトリを `/content/RMT_utils` にclone/pullしてから以下を実行してください。
依存関係（必要な場合のみ別セルで実行）:
`%pip install -r /content/RMT_utils/notebooks/requirements-bulk.txt`
既存Llama/Gemmaノートは変更しません。


In [ ]:
from pathlib import Path
import sys, os, json, hashlib, subprocess, tempfile
import numpy as np
import pandas as pd
import torch

REPO = next((p for p in [Path.cwd(), *Path.cwd().resolve().parents, Path('/content/RMT_utils')]
             if (p/'bulk_diagnostics.py').exists()), None)
if REPO is None:
    raise RuntimeError('RMT_utilsをcloneし、リポジトリまたはnotebooksフォルダで開いてください')
sys.path.insert(0, str(REPO))
# 修正: 固定commitのGemma読込処理に、スカラーTensorの数値化を適用する。
# tail_xminのJSON化エラーのみを修正し、KS計算式や保存済み閾値は変更しない。
import importlib
_diagnostics_path = REPO / 'bulk_diagnostics.py'
_original = '    frame = frame.copy()\n'
_fixed = '    frame = frame.copy()\n    # 修正: Gemmaのtail_xminなどのスカラーTensorを通常の数値へ変換する。\n    # pandas.to_jsonの再帰エラーを防ぎ、閾値・KS・固有値配列の値は変えない。\n    for column in frame.columns:\n        if any(isinstance(value, torch.Tensor) and value.ndim == 0 for value in frame[column]):\n            frame[column] = frame[column].map(\n                lambda value: value.detach().cpu().item()\n                if isinstance(value, torch.Tensor) and value.ndim == 0 else value)\n'
_source = _diagnostics_path.read_text()
if _fixed not in _source:
    if _original not in _source:
        raise RuntimeError('読込処理が想定と異なるため修正を中止しました')
    _diagnostics_path.write_text(_source.replace(_original, _fixed, 1))
import bulk_diagnostics
importlib.reload(bulk_diagnostics)

from bulk_diagnostics import load_metrics, run_diagnostics, SOURCE_RUNS, publish_wandb, resolve_analysis_model
from funcs1 import get_esd_metrics

MODE = 'full'  # synthetic / probe / full
ALLOW_MODEL_DOWNLOAD = True
PUBLISH_WANDB = True
MODEL_KEY = 'gemma'  # llama / gemma（1モデルずつ）
MODEL_REVISION = 'cc012e0a6d0787b4adcc0fa2c4da74402494554d'  # 本番時にHugging Faceのcommit SHAを指定する
SOURCE_RUN_FOR_MODEL = 'ryoya-zushi1210-tokyo-university-of-science/LlaMa-RMT-Analysis/jv0d6kiu'  # 上記2つから、configで対象モデルを確認して指定
SOURCE_CONFIRMED = True  # revision・dtype・保存元モデルを確認してTrue
SVD_DEVICE = 'cuda'  # T4を手動選択後、probeで 'cuda' を評価する
BETA = 0.1
PROBE_LAYERS = 2
DRIVE_DATA = Path('/content/drive/MyDrive/TUS/hashiguchi/data')
EXPERIMENT_TAG = 'bulk-ks-20260925-v1'
MODEL_SPECS = {
    'llama': ('meta-llama/Llama-3.2-3B', 'llama-3.2-3B_esd_metrics.pkl'),
    'gemma': ('google/gemma-3-4b-pt', 'Gemma3-4B_esd_metrics.pkl'),
}
assert MODE in {'synthetic', 'probe', 'full'}
print('Mode:', MODE, '| GPU selection remains a manual VS Code/Colab setting.')


## 入力と認証
DriveはVS Codeの `Colab: Mount Google Drive to Server...` でマウントできます。
本番時はHugging Faceでモデル利用許可を取得し、`huggingface_hub.login()`で認証します。
W&Bは送信前に`wandb.login()`を実行してください。トークンをNotebookに直書きしないでください。
保存元run（対応関係はrunのconfigを表示して確認）:
- https://wandb.ai/ryoya-zushi1210-tokyo-university-of-science/LlaMa-RMT-Analysis/runs/jv0d6kiu
- https://wandb.ai/ryoya-zushi1210-tokyo-university-of-science/LlaMa-RMT-Analysis/runs/vyzq5p34
保存された閾値は変更しません。既存KSは `_saved` 列に保持します。


In [ ]:
# このセルはrunの小さなメタデータを読むだけ。モデルやGPUは取得しません。
INSPECT_SOURCE_RUNS = False
if INSPECT_SOURCE_RUNS:
    import wandb
    wandb.login()
    api = wandb.Api()
    for source_run in SOURCE_RUNS:
        r = api.run(source_run)
        print('Source run:', source_run, 'name:', r.name)
        print({k:v for k,v in r.config.items()
               if k in ('model_id','model_name','model','revision','torch_dtype')})


In [ ]:
if MODE == 'synthetic':
    torch.manual_seed(42)
    model = torch.nn.Sequential(torch.nn.Linear(48, 24, bias=False),
                                torch.nn.Linear(24, 16, bias=False))
    saved = get_esd_metrics(model)
    OUTPUT = Path(tempfile.mkdtemp(prefix='rmt-bulk-smoke-'))
    metadata = {'model_id': 'synthetic/tiny-linear', 'source': 'seed=42', 'synthetic': True}
else:
    if not ALLOW_MODEL_DOWNLOAD or not SOURCE_CONFIRMED or not MODEL_REVISION or SOURCE_RUN_FOR_MODEL not in SOURCE_RUNS:
        raise RuntimeError('本番未承認: MODEL_REVISION・保存元を確認して実行フラグを設定してください')
    model_id, filename = MODEL_SPECS[MODEL_KEY]
    source = DRIVE_DATA/filename
    saved = load_metrics(source)
    OUTPUT = DRIVE_DATA/'bulk_ks'/MODEL_KEY/(EXPERIMENT_TAG+'-'+MODE)
    metadata = {'model_id': model_id, 'model_revision': MODEL_REVISION,
                'weight_dtype': 'float16', 'source_file': filename,
                'source_sha256': hashlib.sha256(source.read_bytes()).hexdigest(),
                'source_run': SOURCE_RUN_FOR_MODEL, 'candidate_source_runs': SOURCE_RUNS, 'synthetic': False}
metadata['git_commit'] = subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip()
metadata['code_sha256'] = {f: hashlib.sha256((REPO/f).read_bytes()).hexdigest()
                           for f in ['funcs1.py','bulk_diagnostics.py']}
metadata['torch_version'] = torch.__version__
metadata['numpy_version'] = np.__version__
print('Input rows:', len(saved), '| output:', OUTPUT)


In [ ]:
# 本番設定の最終確認。ここではモデル取得やSVDは実行しません。
expected_rows = 238
assert MODE == "full"
assert ALLOW_MODEL_DOWNLOAD and SOURCE_CONFIRMED and PUBLISH_WANDB
assert SVD_DEVICE == "cuda"
assert torch.cuda.is_available(), "VS CodeでT4などのGPU Colabランタイムを選択してください"
assert len(saved) == expected_rows, (len(saved), expected_rows)
print({
    "model": MODEL_KEY,
    "mode": MODE,
    "input_rows": len(saved),
    "svd_device": SVD_DEVICE,
    "gpu": torch.cuda.get_device_name(),
    "output": str(OUTPUT),
    "wandb_after_completion": PUBLISH_WANDB,
})


## モデル取得と解析
本番ではモデル全体をCPUメモリに置き、SVD対象の行列だけを指定deviceへ転送します。
SVD用の一時領域も必要です。probeはモデル全体を取得するため、本番許可後にのみ実施します。
float64のSVDはT4で必ず高速になるとは限りません。probeの所要時間とメモリを確認します。
Gemmaは保存済み行列名に完全一致する言語モデル階層を選びます。一意でない場合は停止します。


In [ ]:
if MODE != 'synthetic':
    from transformers import AutoModelForCausalLM, Gemma3ForConditionalGeneration
    cls = Gemma3ForConditionalGeneration if MODEL_KEY == 'gemma' else AutoModelForCausalLM
    loaded = cls.from_pretrained(model_id, revision=MODEL_REVISION,
                                 torch_dtype=torch.float16, device_map='cpu', low_cpu_mem_usage=True)
    model = resolve_analysis_model(loaded, saved.name)
    if SVD_DEVICE == 'cuda':
        if not torch.cuda.is_available():
            raise RuntimeError('CUDA GPU未接続。CPUへ黙って切り替えず停止します')
        print('GPU:', torch.cuda.get_device_name())
        torch.cuda.reset_peak_memory_stats()

results = run_diagnostics(model, saved, OUTPUT, metadata, device=SVD_DEVICE,
                          beta=BETA, limit=PROBE_LAYERS if MODE == 'probe' else None)
if SVD_DEVICE == 'cuda':
    print('Peak allocated GB:', torch.cuda.max_memory_allocated()/1e9)
display(results[['name','KS_preDE','bulkKS_preDE','KS_postDE','bulkKS_postDE',
                 'KS_postDE_1','bulkKS_DE_1']].head(12))


## 結果の比較
全体KS→bulkKSの低下だけではDE効果としません。同じ行列のpre/postを比較します。
除いた固有値の比率も併記し、モデル別・モジュール別に見ます。
人工行列モードの図は動作確認であり、Llama/Gemmaについての研究結果ではありません。


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for ax, full, bulk, title in [
    (axes[0], 'KS_preDE', 'bulkKS_preDE', 'Before DE'),
    (axes[1], 'KS_postDE', 'bulkKS_postDE', 'After DE (BEMA)'),
    (axes[2], 'KS_postDE_1', 'bulkKS_DE_1', 'After DE (variance=1)'),
]:
    ax.scatter(results[full], results[bulk], s=24, alpha=.75, color='#2864a0')
    ax.plot([0,1],[0,1], '--', color='gray')
    ax.set(xlabel='Full KS', ylabel='Bulk KS', title=title, xlim=(0,1), ylim=(0,1))
fig.suptitle(('SYNTHETIC CHECK — ' if MODE=='synthetic' else MODEL_KEY+' — ')+f'n={len(results)} matrices')
fig.savefig(OUTPUT/'full_vs_bulk.png', dpi=140)
plt.show()
fig, ax = plt.subplots(figsize=(6,4), constrained_layout=True)
sc = ax.scatter(results.bulkKS_preDE, results.bulkKS_postDE,
                c=results.bulk_ratio_postDE, vmin=0, vmax=1, cmap='viridis')
ax.plot([0,1],[0,1], '--', color='gray')
ax.set(xlabel='Bulk KS before DE', ylabel='Bulk KS after DE', xlim=(0,1), ylim=(0,1),
       title='Paired matrices: below diagonal = smaller KS after DE')
fig.colorbar(sc, ax=ax, label='Bulk fraction after DE')
fig.savefig(OUTPUT/'paired_bulk.png', dpi=140)
plt.show()
summary = results.groupby('module_type')[['bulkKS_preDE','bulkKS_postDE','bulkKS_DE_1',
                                         'bulkKS_delta_post_minus_pre','bulk_ratio_postDE']].median()
summary.to_csv(OUTPUT/'module_medians.csv')
display(summary)
print('Matrices with finite paired KS:', results[['bulkKS_preDE','bulkKS_postDE']].dropna().shape[0])


## 保存
CSVとmetadataは解析中からDriveの指定フォルダに保存します（人工行列は一時フォルダのみ）。
W&Bは新規runにTableと全結果Artifactを追加します。既存runは更新しません。
再開は同じ設定・同じコード・同じ入力に限ります。設定変更時はEXPERIMENT_TAGを変更してください。


In [ ]:
if PUBLISH_WANDB:
    if MODE != 'full':
        raise RuntimeError('W&Bへの研究結果の送信はfullモードだけで行います')
    receipt = OUTPUT/'wandb_run_url.txt'
    if receipt.exists():
        print('W&B送信済み:', receipt.read_text().strip())
    else:
        url = publish_wandb(results, OUTPUT, metadata, [SOURCE_RUN_FOR_MODEL])
        receipt.write_text(url + '\n')
        print('W&B:', url)
else:
    print('W&B送信なし。結果保存先:', OUTPUT)


## 解釈と次の段階
本番結果は未実行です。完了後、全体KSとbulkKS、DE前後の対応差、バルク比率をモデル別・モジュール別に確認します。小さいbulkKSは閾値処理の整合性を支持する診断であり、シグナルの真偽・TW誤検出率・PPL維持を保証するものではありません。

## 完了後の自動切断

このセルは最終セルです。上の計算・図表保存・W&B送信が正常終了した場合だけ実行され、Driveへの未反映書き込みを同期してからColabランタイムを解放します。
実行後は同じサーバーへ再接続できない場合があるため、再開時は新しいColabサーバーを作成してください。


In [ ]:
# 全結果のDrive保存とW&B送信が終わった後に、Driveを同期してColabを解放します。
from google.colab import drive, runtime

drive.flush_and_unmount()
print("Driveへの書き込みを同期しました。Colabランタイムを解放します。")
runtime.unassign()
